# TransFG — FGVC-Aircraft (CUB 가중치 전이학습)

**논문**: TransFG: A Transformer Architecture for Fine-grained Recognition (AAAI 2022)  
**데이터셋**: FGVC-Aircraft 2013b (항공기 100 variant 분류)

---

## 프로젝트 개요

이 노트북은 기존 `TransFG_FGVC_Aircraft.ipynb`와 동일한 학습을 수행하되,  
**초기 가중치를 CUB-200-2011로 학습된 TransFG checkpoint**로 변경하여  
**Fine-Grained 도메인 간 전이학습 효과**를 검증하는 실험입니다.

### 기존 노트북과의 차이

| 항목 | 기존 `TransFG_FGVC_Aircraft.ipynb` | **본 노트북 (Aircraft_New)** |
|------|-----------------------------------|------------------------------|
| 초기 가중치 | `pretrained/ViT-B_16.npz` (ImageNet-21k) | **`output/transfg_cub_checkpoint.bin`** (CUB 90.84%) |
| 가중치 학습 도메인 | ImageNet-21k (일반 사물 14M장) | **CUB-200-2011 (조류 11K장, fine-grained)** |
| 분류 헤드 | 새로 초기화 (zero_head=True) | 새로 초기화 (CUB 200 → Aircraft 100, 클래스 수 다름) |
| 백본 (Embeddings, Transformer) | ImageNet-21k 가중치 | **CUB-trained TransFG 가중치** |
| 학습 steps | 200 (quick test) | 200 (quick test) — 동일 비교 |
| 그 외 모든 하이퍼파라미터 | — | 기존과 동일 |

### 검증하려는 가설

> **가설**: Fine-grained 인식에 특화된 CUB-trained 가중치가  
> 일반 ImageNet-21k 가중치보다 Aircraft 분류에 더 빠르게 수렴할 것이다.

### 비교 대상 (Baseline)

기존 `TransFG_FGVC_Aircraft.ipynb`의 200 steps 결과:

| Step | Val Accuracy |
|------|-------------|
| 50 | 1.29% |
| 100 | 2.13% |
| 150 | 6.24% |
| **200** | **6.57%** |

본 노트북 실행 후 위 수치와 비교하여:
- **CUB > 6.57%** → 전이학습 효과 있음 (fine-grained 특성 공유)
- **CUB ≈ 6.57%** → 효과 미미 (도메인 차이로 상쇄)
- **CUB < 6.57%** → 오히려 방해 (조류 특화 feature가 항공기 인식 저해)

### 도메인 차이 (참고)

| | CUB-200-2011 | FGVC-Aircraft |
|---|---|---|
| 대상 | 조류 (유기체) | 항공기 (인공물) |
| 텍스처 | 깃털, 부드러운 곡선 | 금속 표면, 직선/평면 |
| 차별화 포인트 | 부리·날개·깃털 패턴 | 동체 길이·날개 후퇴각 |
| 클래스 | 200종 | 100 variant |

도메인이 다르지만 **둘 다 미세 차이 식별이 필요한 fine-grained 문제**이므로,  
TransFG의 PSM(Part Selection Module) 학습 패턴이 전이될 가능성이 있습니다.

## 1. 환경 확인

In [ ]:
import sys
import torch

print(f"Python : {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

## 2. 모듈 임포트 및 경로 설정

기존 노트북과 동일하지만 `CUB_CHECKPOINT` 경로가 추가됩니다.

In [ ]:
import sys
import random
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

# TransFG 내부 모듈
TRANSFG_ROOT = Path("TransFG").resolve()
if str(TRANSFG_ROOT) not in sys.path:
    sys.path.insert(0, str(TRANSFG_ROOT))

from models.modeling import VisionTransformer, CONFIGS

# FGVC-Aircraft 전용 모듈
from dataset_aircraft      import AircraftDataset
from data_loader_aircraft  import get_aircraft_loaders

# 공통 유틸
from trainer         import train, validate
from inference_utils import predict_single, predict_batch, evaluate_dataset
from visualization   import show_sample_grid, plot_history, visualize_predictions, visualize_attention

DEVICE         = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_ROOT      = Path("data/FGVC_Aircraft/fgvc-aircraft-2013b")
PRETRAINED     = Path("pretrained/ViT-B_16.npz")                 # 참고용 (이번엔 미사용)
CUB_CHECKPOINT = Path("output/transfg_cub_checkpoint.bin")        # ← 핵심: CUB 학습 가중치
OUTPUT_DIR     = Path("output/fgvc_aircraft_from_cub")            # 별도 디렉토리
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device         : {DEVICE}")
print(f"DATA_ROOT      : {DATA_ROOT.resolve()}")
print(f"CUB_CHECKPOINT : {CUB_CHECKPOINT.resolve()}")
print(f"OUTPUT_DIR     : {OUTPUT_DIR.resolve()}")

assert DATA_ROOT.exists(),      "데이터셋 없음 — data/FGVC_Aircraft/fgvc-aircraft-2013b 확인"
assert CUB_CHECKPOINT.exists(), f"CUB 가중치 없음 — {CUB_CHECKPOINT} (먼저 CUB 학습 완료 필요)"
print(f"\nCUB 가중치 크기: {CUB_CHECKPOINT.stat().st_size/1e6:.0f} MB")

# 한글 폰트 설정
import matplotlib.font_manager as _fm
_font = Path("NotoSansCJK-Regular.ttc")
if _font.exists():
    _fm.fontManager.addfont(str(_font.resolve()))
    plt.rcParams["font.family"] = "Noto Sans CJK JP"

## 3. 데이터셋 탐색

In [ ]:
from collections import Counter

trainset_raw = AircraftDataset(root=str(DATA_ROOT), split="train", level="variant", transform=None)
valset_raw   = AircraftDataset(root=str(DATA_ROOT), split="val",   level="variant", transform=None)
testset_raw  = AircraftDataset(root=str(DATA_ROOT), split="test",  level="variant", transform=None)

print(f"Train : {len(trainset_raw):>5}장")
print(f"Val   : {len(valset_raw):>5}장")
print(f"Test  : {len(testset_raw):>5}장")
class_names = trainset_raw.get_class_names()
print(f"클래스: {len(class_names)}종")
print(f"\n클래스 예시 (5개):")
for c in class_names[:5]:
    print(f"  - {c}")

## 4. 하이퍼파라미터 설정

**기존 Aircraft 노트북과 100% 동일** (공정한 비교를 위해).

In [ ]:
CFG = {
    "model_type"   : "ViT-B_16",
    "split"        : "overlap",
    "slide_step"   : 12,
    "img_size"     : 448,
    "num_classes"  : 100,          # FGVC-Aircraft variant: 100종 (CUB는 200)
    "train_batch"  : 8,
    "eval_batch"   : 8,
    "num_steps"    : 10000,
    "quick_steps"  : 200,
    "lr"           : 3e-2,
    "warmup_steps" : 500,
    "decay_type"   : "cosine",
    "smoothing"    : 0.0,
    "fp16"         : True,
    "run_name"     : "transfg_aircraft_from_cub",   # CUB 가중치 사용 표시
    "num_workers"  : 16,                            # 빠른 데이터 로딩
    "level"        : "variant",
    "use_trainval" : False,
}

print("=== TransFG FGVC-Aircraft (CUB init) Config ===")
for k, v in CFG.items():
    print(f"  {k:20s}: {v}")

## 5. DataLoader 생성

In [ ]:
train_loader, val_loader, test_loader, trainset, valset, testset = get_aircraft_loaders(
    data_root        = str(DATA_ROOT),
    train_batch_size = CFG["train_batch"],
    eval_batch_size  = CFG["eval_batch"],
    img_size         = CFG["img_size"],
    num_workers      = CFG["num_workers"],
    level            = CFG["level"],
    use_trainval     = CFG["use_trainval"],
)

x_batch, y_batch = next(iter(train_loader))
print(f"Input batch  : {x_batch.shape}")
print(f"Label batch  : {y_batch.shape}")
print(f"Train 배치 수: {len(train_loader)}")
print(f"Val   배치 수: {len(val_loader)}")
print(f"Test  배치 수: {len(test_loader)}")

## 6. 모델 생성 및 CUB 가중치 로드 ⭐

**이 노트북의 핵심 차이점.**

### 가중치 로드 전략

1. **모델 생성**: `num_classes=100` (Aircraft) 으로 빈 모델 생성
2. **CUB checkpoint 로드**: `strict=False` 로 로드
   - **로드되는 것**: Embeddings, Transformer (12 layers), Part Selection Module, LayerNorm
   - **로드 안 되는 것**: `part_head` (CUB 200 vs Aircraft 100 — 크기 불일치)
3. **분류 헤드**: PyTorch 기본 초기화 (랜덤) 후 학습으로 fit

### 왜 strict=False?

CUB checkpoint의 `part_head.weight`는 `[200, 768]`, 새 모델의 `part_head.weight`는 `[100, 768]` 이라 그대로 로드 불가.  
`strict=False`로 호환되는 backbone만 로드하고, 헤드는 처음부터 학습.

In [ ]:
# Step 1: 모델 생성 (Aircraft 100 classes)
config = CONFIGS[CFG["model_type"]]
config.split      = CFG["split"]
config.slide_step = CFG["slide_step"]

p_size = config.patches["size"][0]
n_side = (CFG["img_size"] - p_size) // CFG["slide_step"] + 1
print(f"patch grid : {n_side}×{n_side} = {n_side**2} patches\n")

model = VisionTransformer(
    config,
    img_size        = CFG["img_size"],
    zero_head       = True,
    num_classes     = CFG["num_classes"],   # 100 (Aircraft)
    smoothing_value = CFG["smoothing"],
)

# Step 2: CUB checkpoint 로드 (head 가중치 제외)
print(f"CUB checkpoint 로드 중: {CUB_CHECKPOINT}")
ckpt = torch.load(str(CUB_CHECKPOINT), map_location="cpu")
cub_state_dict = ckpt["model"]

# CUB checkpoint 정보 출력
print(f"  - CUB 가중치 키 개수    : {len(cub_state_dict)}")
print(f"  - CUB 학습 step         : {ckpt.get('global_step', 'N/A')}")
print(f"  - CUB 최고 정확도       : {ckpt.get('best_acc', 0.0):.4f}")

# 크기 불일치 layer 제거 (part_head: 200 → 100)
filtered_state_dict = {
    k: v for k, v in cub_state_dict.items()
    if not k.startswith("part_head")
}
removed_keys = set(cub_state_dict.keys()) - set(filtered_state_dict.keys())
print(f"\n  - head 가중치 제외 (CUB 200 vs Aircraft 100):")
for k in removed_keys:
    print(f"      · {k}  (제외됨)")

# strict=False 로 로드 (head는 새로 초기화된 상태 유지)
missing, unexpected = model.load_state_dict(filtered_state_dict, strict=False)

print(f"\n  - missing (모델에는 있지만 ckpt에서 로드 안 된 키): {len(missing)}")
for k in missing:
    print(f"      · {k}")
print(f"  - unexpected (ckpt에 있지만 모델에 없는 키)     : {len(unexpected)}")
for k in unexpected:
    print(f"      · {k}")

model = model.to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n학습 파라미터: {n_params/1e6:.1f}M")
print(f"분류 헤드    : {model.part_head}  ← 새로 초기화됨 (zero_head=True)")

# 동작 확인
model.eval()
with torch.no_grad():
    out = model(torch.randn(2, 3, CFG["img_size"], CFG["img_size"]).to(DEVICE))
print(f"추론 출력    : {out.shape}  (배치=2, 클래스=100)")

## 7. 빠른 학습 테스트 (200 steps)

기존 노트북과 **동일 설정**으로 200 steps 학습.  
결과를 기존 ImageNet-21k 베이스라인 (val_acc=6.57%) 과 비교합니다.

In [ ]:
import time

print(f"Quick training: {CFG['quick_steps']} steps  |  init: CUB checkpoint  |  eval: val_loader")
print("=" * 70)

start = time.time()
best_acc, history = train(
    model        = model,
    train_loader = train_loader,
    test_loader  = val_loader,    # quick test에서는 val로 검증
    device       = DEVICE,
    num_steps    = CFG["quick_steps"],
    learning_rate= CFG["lr"],
    warmup_steps = 50,
    decay_type   = CFG["decay_type"],
    eval_every   = 50,
    output_dir   = str(OUTPUT_DIR),
    run_name     = CFG["run_name"],
    fp16         = CFG["fp16"],
    gradient_accumulation_steps = 1,
    max_grad_norm = 1.0,
)
elapsed = time.time() - start
print(f"\n경과 시간    : {elapsed:.1f}초 ({elapsed/60:.1f}분)")
print(f"Best val_acc : {best_acc:.4f} ({best_acc*100:.2f}%)")

## 8. 학습 곡선 시각화

In [ ]:
plot_history(history, eval_every=50)

## 9. ImageNet-21k 베이스라인과 비교

동일한 200 steps 학습 결과를 비교.

In [ ]:
# 기존 ImageNet-21k 베이스라인 (TransFG_FGVC_Aircraft.ipynb 결과)
baseline_imagenet21k = {
    50  : 0.0129,
    100 : 0.0213,
    150 : 0.0624,
    200 : 0.0657,
}

# 본 노트북 결과 (CUB checkpoint)
cub_init_results = {}
for i, acc in enumerate(history["val_acc"]):
    step = (i + 1) * 50
    cub_init_results[step] = acc

# 비교 출력
print("=" * 60)
print(f"{'Step':>6} | {'ImageNet-21k':>15} | {'CUB checkpoint':>15} | {'차이':>8}")
print("-" * 60)
for step in [50, 100, 150, 200]:
    base = baseline_imagenet21k.get(step, 0.0)
    cub  = cub_init_results.get(step, 0.0)
    diff = cub - base
    sign = "+" if diff >= 0 else ""
    print(f"{step:>6} | {base*100:>13.2f}%  | {cub*100:>13.2f}%  | {sign}{diff*100:>6.2f}%p")
print("=" * 60)

# 결론
final_diff = cub_init_results.get(200, 0.0) - baseline_imagenet21k[200]
print("\n[결론]")
if final_diff > 0.02:
    print(f"  CUB 가중치 사용 시 {final_diff*100:+.2f}%p 향상 → fine-grained 전이학습 효과 확인")
elif final_diff > -0.02:
    print(f"  차이 미미 ({final_diff*100:+.2f}%p) → 도메인 차이로 효과 상쇄")
else:
    print(f"  CUB 가중치 사용 시 {final_diff*100:+.2f}%p 하락 → 조류 특화 feature가 항공기 인식 저해")

In [ ]:
# 시각화 비교
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
steps = sorted(baseline_imagenet21k.keys())

ax.plot(steps, [baseline_imagenet21k[s]*100 for s in steps], 'o-', label='ImageNet-21k init', linewidth=2, markersize=8)

if cub_init_results:
    cub_steps = sorted(cub_init_results.keys())
    ax.plot(cub_steps, [cub_init_results[s]*100 for s in cub_steps], 's-', label='CUB checkpoint init', linewidth=2, markersize=8)

ax.set_xlabel("Training Step")
ax.set_ylabel("Validation Accuracy (%)")
ax.set_title("FGVC-Aircraft 200 steps 학습 비교 — 초기 가중치 효과")
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. 전체 학습 (Optional, 10,000 steps)

기존 노트북과 동일한 방식. 200 steps 결과가 유의미하면 전체 학습 진행.

In [ ]:
# 전체 학습 — 주석 해제 후 실행
# import time
#
# # 모델 재생성 + CUB 가중치 재로드 (head 제외)
# model = VisionTransformer(config, img_size=CFG["img_size"],
#                           zero_head=True, num_classes=CFG["num_classes"],
#                           smoothing_value=CFG["smoothing"])
# ckpt = torch.load(str(CUB_CHECKPOINT), map_location="cpu")
# filtered_state_dict = {k: v for k, v in ckpt["model"].items() if not k.startswith("part_head")}
# model.load_state_dict(filtered_state_dict, strict=False)
# model = model.to(DEVICE)
#
# start = time.time()
# best_acc, history_full = train(
#     model        = model,
#     train_loader = train_loader,
#     test_loader  = val_loader,
#     device       = DEVICE,
#     num_steps    = CFG["num_steps"],          # 10,000
#     learning_rate= CFG["lr"],
#     warmup_steps = CFG["warmup_steps"],       # 500
#     decay_type   = CFG["decay_type"],
#     eval_every   = 100,
#     output_dir   = str(OUTPUT_DIR),
#     run_name     = CFG["run_name"],
#     fp16         = CFG["fp16"],
#     gradient_accumulation_steps = 1,
#     max_grad_norm = 1.0,
# )
# print(f"\n경과 시간: {(time.time()-start)/3600:.2f}시간")
# print(f"Best val_acc: {best_acc:.4f}")

## 요약

| 셀 | 내용 |
|---|---|
| 0 | 프로젝트 개요 — 기존 노트북과의 차이 |
| 1~2 | 환경 확인 |
| 3~4 | 모듈 임포트, 경로 설정 (CUB checkpoint 추가) |
| 5~6 | 데이터셋 탐색 |
| 7~8 | 하이퍼파라미터 설정 (기존과 동일) |
| 9~10 | DataLoader 생성 |
| 11~12 | **모델 생성 + CUB 가중치 로드 (strict=False)** ⭐ |
| 13~14 | 빠른 학습 테스트 (200 steps) |
| 15~16 | 학습 곡선 시각화 |
| 17~19 | **ImageNet-21k 베이스라인과 비교** ⭐ |
| 20~21 | 전체 학습 (Optional, 10k steps) |